# Predicción de Informalidad Laboral en Colombia: Ingesta de Datos
**Fuente:** Gran Encuesta Integrada de Hogares (GEIH) — DANE 2024  
**Entorno:** Local — Python + DuckDB

---
### SetUp

Instalación e importe de librerías.

Estructura de archivos esperada en el directorio del proyecto:
```
Proyecto_Final_maestria/
├── 2024_data/
│   ├── Caracteristicas_Generales/
│   │   ├── 1.Caracteristicas_Generales_Enero.csv
│   │   ├── 2.Caracteristicas_Generales_Febrero.csv
│   │   └── ...  (12 archivos, uno por mes)
│   └── Ocupados/
│       ├── 1.Ocupados_Enero.csv
│       ├── 2.Ocupados_Febrero.csv
│       └── ...  (12 archivos, uno por mes)
├── parquet/           ← salida de este notebook
└── geih_2024.duckdb   ← base de datos DuckDB de salida
```

In [ ]:
#!pip install duckdb pyarrow --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import duckdb
from pathlib import Path

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


---
### Extracción de datos

In [3]:
# Rutas locales — ajustar BASE_DIR si el notebook se ejecuta desde otra ubicación
BASE_DIR          = Path('.')
CARPETA_GENERALES = BASE_DIR / '2024_data' / 'Caracteristicas_Generales'
CARPETA_OCUPADOS  = BASE_DIR / '2024_data' / 'Ocupados'
CARPETA_SALIDA    = BASE_DIR / 'parquet'
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

SEPARADOR = ';'
ENCODING  = 'latin-1'

for carpeta, nombre in [(CARPETA_GENERALES, 'Características Generales'),
                        (CARPETA_OCUPADOS,  'Ocupados')]:
    if carpeta.exists():
        archivos = sorted(carpeta.glob('*.csv'))
        print(f'{nombre}: {len(archivos)} archivos CSV')
        for a in archivos:
            print(f'  {a.name}')
    else:
        print(f'Carpeta no encontrada: {carpeta}')

Características Generales: 12 archivos CSV
  1.Caracteristicas_Generales_Enero.csv
  10.Caracteristicas_Generales_Octube.csv
  11.Caracteristicas_Generales_Noviembre.csv
  12.Caracteristicas_Generales_Diciembre.csv
  2.Caracteristicas_Generales_Febrero.csv
  3.Caracteristicas_Generales_Marzo.csv
  4.Caracteristicas_Generales_Abril.csv
  5.Caracteristicas_Generales_Mayo.csv
  6.Caracteristicas_Generales_Junio.csv
  7.Caracteristicas_Generales_Julio.csv
  8.Caracteristicas_Generales_Agosto.csv
  9.Caracteristicas_Generales_Septiembre.csv
Ocupados: 12 archivos CSV
  1.Ocupados_Enero.csv
  10.Ocupados_Octubre.csv
  11.Ocupados_Noviembre.csv
  12.Ocupados_Diciembre.csv
  2.Ocupados_Febrero.csv
  3.Ocupados_Marzo.csv
  4.Ocupados_Abril.csv
  5.Ocupados_Mayo.csv
  6.Ocupados_Junio.csv
  7.Ocupados_Julio.csv
  8.Ocupados_Agosto.csv
  9.Ocupados_Septiembre.csv


#### Columnas a cargar

Se cargan las columnas consideradas relevantes y suficientes para un volumen de datos adecuado y razonable para el alcance relativamente corto del proyecto. Un solo archivo de Características Generales en su estado natural contiene 55 columnas y, en promedio, 70000 filas. Para un total de casi 4 millones de datos por archivo. 

**Módulo Características Generales:**

| Variable | Descripción |
|---|---|
| `P3271` | Sexo (1=Hombre, 2=Mujer) |
| `P6040` | Edad en años cumplidos |
| `P3042` | Nivel educativo (1=Ninguno a 13=Doctorado) |
| `P3042S1` | Años aprobados en ese nivel |
| `P6070` | Estado civil |
| `DPTO` | Código del departamento (33 dominios DANE) |
| `CLASE` | Zona (1=Cabecera municipal, 2=Rural) |
| `FEX_C18` | Factor de expansión estadística |

**Módulo Ocupados:**

| Variable | Descripción |
|---|---|
| **`P6920`** | **Variable objetivo** (1=Cotizante, 2=No cotizante, 3=Pensionado) |
| `P6430` | Posición ocupacional |
| `P6450` | Tipo de contrato (verbal, escrito, no sabe) |
| `P6460` | Naturaleza del contrato (indefinido, fijo, NS/NR) |
| `P6800` | Horas trabajadas en la semana |
| `P3069` | Tamaño del establecimiento |
| `RAMA2D_R4` | Rama de actividad económica (CIIU 2 dígitos) |
| `P7040` | Pluriempleo (1=Sí, 2=No) |
| `INGLABO` | Ingreso laboral (solo referencia descriptiva) |

In [4]:
# Llave única que identifica a cada persona en la encuesta
LLAVES = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']

COLS_GENERALES = LLAVES + [
    'P3271',    # Sexo (1=Hombre, 2=Mujer)
    'P6040',    # Edad en años cumplidos
    'P3042',    # Nivel educativo (1=Ninguno a 13=Doctorado)
    'P3042S1',  # Años aprobados en ese nivel educativo
    'P6070',    # Estado civil
    'DPTO',     # Código del departamento (33 dominios DANE)
    'CLASE',    # Zona (1=Cabecera municipal, 2=Rural)
    'FEX_C18',  # Factor de expansión (peso estadístico)
]

COLS_OCUPADOS = LLAVES + [
    'P6920',     # ← VARIABLE OBJETIVO
                 #   1 = Cotizante a salud por trabajo      → FORMAL
                 #   2 = No cotizante                       → INFORMAL
                 #   3 = Pensionado (retirado)
    'P6430',     # Posición ocupacional (empleado, cuenta propia, patrón...)
    'P6450',     # Tipo de contrato (verbal, escrito, no sabe)
    'P6460',     # Naturaleza del contrato (indefinido, fijo, NS/NR)
    'P6800',     # Horas trabajadas en la semana de referencia
    'P3069',     # Tamaño del establecimiento (1 persona a 201 o más)
    'RAMA2D_R4', # Rama de actividad económica (CIIU a 2 dígitos)
    'P7040',     # Pluriempleo (1=Sí, 2=No)
    'INGLABO',   # Ingreso laboral
]

# Validar que las columnas existan en el primer archivo de cada módulo
archivos_gen = sorted(CARPETA_GENERALES.glob('*.csv'))
archivos_ocu = sorted(CARPETA_OCUPADOS.glob('*.csv'))

cols_gen = pd.read_csv(archivos_gen[0], nrows=0, encoding=ENCODING, sep=SEPARADOR).columns.tolist()
cols_ocu = pd.read_csv(archivos_ocu[0], nrows=0, encoding=ENCODING, sep=SEPARADOR).columns.tolist()

print(f'Verificando: {archivos_gen[0].name}')
todo_ok = True
for col in COLS_GENERALES:
    ok = col in cols_gen
    if not ok: todo_ok = False
    print(f'  {"OK  " if ok else "FALTA"} {col}')

print(f'\nVerificando: {archivos_ocu[0].name}')
for col in COLS_OCUPADOS:
    ok = col in cols_ocu
    if not ok: todo_ok = False
    print(f'  {"OK  " if ok else "FALTA"} {col}')

if todo_ok:
    print('\nTodas las columnas encontradas.')
else:
    print('\nAlgunas columnas no se encontraron.')

print(f'\nColumnas Características Generales: {len(COLS_GENERALES)}')
print(f'Columnas Ocupados:                  {len(COLS_OCUPADOS)}')

Verificando: 1.Caracteristicas_Generales_Enero.csv
  OK   DIRECTORIO
  OK   SECUENCIA_P
  OK   ORDEN
  OK   P3271
  OK   P6040
  OK   P3042
  OK   P3042S1
  OK   P6070
  OK   DPTO
  OK   CLASE
  OK   FEX_C18

Verificando: 1.Ocupados_Enero.csv
  OK   DIRECTORIO
  OK   SECUENCIA_P
  OK   ORDEN
  OK   P6920
  OK   P6430
  OK   P6450
  OK   P6460
  OK   P6800
  OK   P3069
  OK   RAMA2D_R4
  OK   P7040
  OK   INGLABO

Todas las columnas encontradas.

Columnas Características Generales: 11
Columnas Ocupados:                  12


Existe la posibilidad de que año a año cambien los nombres o las descripciones de las variables. Tener en cuenta para futuros reentrenos.

---
### Carga de archivos y unión de módulos

Esta celda puede tardar entre 1 y 5 minutos dependiendo del hardware (12 meses × 2 módulos).

In [5]:
def cargar_modulo(archivos, columnas, nombre):
    dfs = []
    print(f'Cargando: {nombre}')
    for archivo in sorted(archivos):
        cols_disponibles = pd.read_csv(
            archivo, nrows=0, encoding=ENCODING, sep=SEPARADOR
        ).columns.tolist()
        cols_a_leer = [c for c in columnas if c in cols_disponibles]

        df_mes = pd.read_csv(
            archivo,
            usecols=cols_a_leer,
            encoding=ENCODING,
            sep=SEPARADOR,
            low_memory=False
        )
        dfs.append(df_mes)
        print(f'  {archivo.name}: {len(df_mes):,} filas')

    resultado = pd.concat(dfs, ignore_index=True)
    print(f'  → Total {nombre}: {len(resultado):,} filas\n')
    return resultado


df_gen = cargar_modulo(archivos_gen, COLS_GENERALES, 'Características Generales')
df_ocu = cargar_modulo(archivos_ocu, COLS_OCUPADOS,  'Ocupados')

df = pd.merge(df_gen, df_ocu, on=LLAVES, how='inner')

print(f'  Características Generales: {len(df_gen):,} personas')
print(f'  Ocupados:                  {len(df_ocu):,} personas')
print(f'  Después del join (inner):  {len(df):,} registros')
print(f'  Columnas:                  {df.shape[1]}')

del df_gen, df_ocu
print('\nMódulos unidos correctamente')

Cargando: Características Generales
  1.Caracteristicas_Generales_Enero.csv: 67,857 filas
  10.Caracteristicas_Generales_Octube.csv: 68,710 filas
  11.Caracteristicas_Generales_Noviembre.csv: 68,582 filas
  12.Caracteristicas_Generales_Diciembre.csv: 66,559 filas
  2.Caracteristicas_Generales_Febrero.csv: 67,953 filas
  3.Caracteristicas_Generales_Marzo.csv: 68,083 filas
  4.Caracteristicas_Generales_Abril.csv: 68,253 filas
  5.Caracteristicas_Generales_Mayo.csv: 68,421 filas
  6.Caracteristicas_Generales_Junio.csv: 67,856 filas
  7.Caracteristicas_Generales_Julio.csv: 68,436 filas
  8.Caracteristicas_Generales_Agosto.csv: 68,577 filas
  9.Caracteristicas_Generales_Septiembre.csv: 68,263 filas
  → Total Características Generales: 817,550 filas

Cargando: Ocupados
  1.Ocupados_Enero.csv: 28,317 filas
  10.Ocupados_Octubre.csv: 30,807 filas
  11.Ocupados_Noviembre.csv: 30,939 filas
  12.Ocupados_Diciembre.csv: 29,611 filas
  2.Ocupados_Febrero.csv: 29,211 filas
  3.Ocupados_Marzo.csv: 29

---
### Construcción de variable objetivo `INFORMAL`

| P6920 | Significado | INFORMAL |
|---|---|---|
| 1 | Cotizante a salud por su trabajo | **0** (formal) |
| 2 | No cotizante | **1** (informal) |
| 3 | Pensionado | **NaN** |

Optamos por eliminar la tercera variante de la variable P6920, ya que es un caso especial el cuál podría introducir sesgo al modelo. Los pensionados ocupados son personas que se salen completamente del foco de la investigación. Al representar más o menos un 2% de los datos totales, no es un impacto negativo significativo.

In [6]:
# 1 = Formal
# 2 = Informal
# 3 o Null = NaN (pensionados — se excluyen del modelo)
df['INFORMAL'] = df['P6920'].map({1: 0, 2: 1})

total      = len(df)
n_formal   = (df['INFORMAL'] == 0).sum()
n_informal = (df['INFORMAL'] == 1).sum()
n_nulos    = df['INFORMAL'].isna().sum()

print('=== DISTRIBUCIÓN DE INFORMAL ===')
print(f'  INFORMAL = 0 (formal):      {n_formal:,}  ({n_formal/total*100:.1f}%)')
print(f'  INFORMAL = 1 (informal):    {n_informal:,}  ({n_informal/total*100:.1f}%)')
print(f'  INFORMAL = NaN (pensionado/nulo): {n_nulos:,}  ({n_nulos/total*100:.1f}%)')
print(f'  Total filas:                {total:,}')

=== DISTRIBUCIÓN DE INFORMAL ===
  INFORMAL = 0 (formal):      150,925  (42.2%)
  INFORMAL = 1 (informal):    200,796  (56.1%)
  INFORMAL = NaN (pensionado/nulo): 6,308  (1.8%)
  Total filas:                358,029


---
### Guardado y creación de base de datos

Guardamos el dataset en Parquet (mucho más liviano y rápido que CSV) y también en DuckDB para consultas SQL analíticas.

In [7]:
ruta_parquet = CARPETA_SALIDA / 'geih_2024_crudo.parquet'
ruta_duckdb  = BASE_DIR / 'geih_2024.duckdb'

# Guardar Parquet
df.to_parquet(ruta_parquet, index=False)
mb_parquet = ruta_parquet.stat().st_size / (1024 * 1024)
print(f'Parquet guardado: {ruta_parquet}')
print(f'   Tamaño:   {mb_parquet:.1f} MB')
print(f'   Filas:    {len(df):,}')
print(f'   Columnas: {df.shape[1]}')

# Crear / actualizar tabla en DuckDB
con = duckdb.connect(str(ruta_duckdb))
con.execute(f"""
    CREATE OR REPLACE TABLE geih AS
    SELECT * FROM read_parquet('{ruta_parquet.as_posix()}')
""")
total_db = con.execute('SELECT COUNT(*) FROM geih').fetchone()[0]
print(f'\nTabla DuckDB "geih" creada: {total_db:,} registros')
print(f'   Archivo: {ruta_duckdb}')
con.close()

print()
print('Resumen final del dataset:')
print(f'  • {len(df):,} filas  |  {df.shape[1]} columnas')
print(f'  • Parquet: {mb_parquet:.1f} MB')
print()
print('Comando de carga para el siguiente notebook:')
print(f"  df = pd.read_parquet('{ruta_parquet}')")

Parquet guardado: parquet\geih_2024_crudo.parquet
   Tamaño:   6.2 MB
   Filas:    358,029
   Columnas: 21

Tabla DuckDB "geih" creada: 358,029 registros
   Archivo: geih_2024.duckdb

Resumen final del dataset:
  • 358,029 filas  |  21 columnas
  • Parquet: 6.2 MB

Comando de carga para el siguiente notebook:
  df = pd.read_parquet('parquet\geih_2024_crudo.parquet')
